# Pipeline Schubert — comunidad curada de 17 taxones (modelo v2)

Ejecuta las celdas en orden desde esta carpeta. Lo costoso (muestreo, controles) queda en `cache/` y `resultados/` y se reanuda si se interrumpe.

**Necesitas en esta carpeta**
- `community17_v2_curated.xml` (modelo curado; el acoplamiento se aplica al cargar), **o** bien reconstruirlo con la sección 0;
- todos los `.py` de esta carpeta, `tolerancia.csv`, `curacion_previa/` y `curacion_v2/`.

Solver: HiGHS (`hybrid`, gratuito). Tiempos aproximados con 2 núcleos: muestreo 2 × 150 estados ≈ 2 h; cada réplica de control ≈ 25 min; dependencias ≈ 10 min.

## 0. (Opcional) Reconstruir el modelo v2 desde los modelos individuales
Necesita `indiv/*.xml` (7 CarveMe + iSRU780 + iHQW-TK) y `community12_curated.xml` (comunidad previa curada).

In [ ]:
RECONSTRUIR = False
import subprocess, sys
if RECONSTRUIR:
    subprocess.run([sys.executable, "inspeccion.py"], check=True)      # diagnóstico de cada modelo
    subprocess.run([sys.executable, "curar_indiv2.py"], check=True)    # oxidasas + ciclos EGC
    subprocess.run([sys.executable, "ensamblar.py"], check=True)       # inserta en la comunidad

## 1. Cargar el modelo y comprobar el óptimo

In [ ]:
import os, glob, pickle, numpy as np, pandas as pd
import pipeline_schubert as ps, curar_modelo as cm
from config_v2 import cargar_comunidad
os.makedirs("resultados", exist_ok=True); os.makedirs("figs", exist_ok=True); os.makedirs("cache", exist_ok=True)
m, mapa = cargar_comunidad()          # M9 + glicerol + trazas Asn/sirohemo, acoplamiento c=400, u=0.01
print("crecimiento comunitario:", round(m.slim_optimize(), 4))
print("E: m =", mapa.m, "| niveles de la bandera:", ps.dimensiones_bandera(mapa))
del m                                  # libera memoria antes del muestreo

## 2. Muestreo random-FBA (150 estados por escenario, lotes de 40 en procesos nuevos)

In [ ]:
for esc in ["SS19", "SS37"]:
    while True:
        f = "cache/flujos_%s_n150.npz" % esc
        n = len(np.load(f)["F"]) if os.path.exists(f) else 0
        print(esc, n, "estados")
        if n >= 150:
            break
        subprocess.run([sys.executable, "lote.py", esc], check=True)

## 3. Mapa `tex` (análisis de sensibilidad) y análisis principal
H1a (cruda / d / ν / z / ν|d), sensibilidad de la bandera, barrido de ε, regresión ordinal y homología persistente.

In [ ]:
if not os.path.exists("mapa_tex.pkl"):
    subprocess.run([sys.executable, "mapa_tex.py"], check=True)
%run analisis.py
%run variantes.py

## 4. Control de perturbación aleatoria (una réplica por proceso)

In [ ]:
B_CTRL = 10
for rep in range(B_CTRL):
    subprocess.run([sys.executable, "control_run.py", str(rep)], check=True)
print(sorted(glob.glob("resultados/control_rep*.csv")))

## 5. Intercambio metabólico (cross-feeding)

In [ ]:
%run intercambio.py
%run ecologia.py
subprocess.run([sys.executable, "ecologia_dep.py"], check=True)
display(pd.read_csv("resultados/eco_dependencias.csv").round(3))

## 6. Figuras, apéndices y manuscrito
Figuras con letras (A), (B)… fuera del panel. `rellenar_v4.py` completa la tabla de escenarios con los controles y escribe `manuscript_v4.tex`.

In [ ]:
%run figuras_run.py
%run fig_ecologia.py
%run suplementario_v2.py
%run a_apendice.py
%run rellenar_v4.py